# OpenAI Agents SDK: 첫 번째 에이전트 프레임워크 프로젝트

이번 노트북에서는 **OpenAI Agents SDK**를 활용하여 멀티 에이전트 시스템을 구축합니다. 이전 노트북에서 직접 구현했던 Tool Use와 에이전트 루프를, 프레임워크가 어떻게 간소화하는지 비교하며 학습합니다.

## 개요

| 주제 | 내용 |
|------|------|
| Agents SDK 소개 | OpenAI의 에이전트 프레임워크 핵심 개념 |
| 에이전트 워크플로우 | 여러 에이전트를 생성하고 병렬 실행 |
| @function_tool | 데코레이터 기반 도구 정의 (JSON 스키마 자동 생성) |
| 에이전트를 도구로 | as_tool()로 에이전트 간 협업 |
| Handoff | 에이전트 간 제어 전달 |
| Trace | OpenAI 트레이스로 실행 흐름 모니터링 |

## 학습 목표

1. OpenAI Agents SDK의 핵심 개념(Agent, Runner, Tool, Handoff) 이해하기
2. 여러 에이전트를 병렬로 실행하고 결과를 비교/선택하기
3. `@function_tool` 데코레이터로 간편하게 도구 정의하기
4. 에이전트 간 협업 패턴(도구 vs 핸드오프) 비교하기
5. Trace를 활용한 에이전트 실행 흐름 모니터링

---

## 이전 노트북과의 비교

```
┌─────────────────────────────────────────────────────────────────────┐
│               직접 구현 vs Agents SDK 비교                         │
├──────────────────────────────┬──────────────────────────────────────┤
│     이전 (직접 구현)          │     이번 (Agents SDK)               │
├──────────────────────────────┼──────────────────────────────────────┤
│  JSON 스키마 직접 작성        │  @function_tool 데코레이터 사용      │
│  handle_tool_calls() 구현    │  SDK가 자동 처리                     │
│  while 루프 직접 관리         │  Runner.run()이 자동 실행            │
│  메시지 히스토리 수동 관리    │  SDK가 컨텍스트 자동 관리            │
│  에이전트 간 통신 직접 구현   │  Handoff로 간편하게 위임             │
└──────────────────────────────┴──────────────────────────────────────┘
```

---

## 1. OpenAI Agents SDK 핵심 개념

OpenAI Agents SDK는 몇 가지 핵심 프리미티브로 구성되어 있습니다:

```
┌─────────────────────────────────────────────────────────────────────┐
│                    Agents SDK 핵심 구성 요소                        │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  ┌─────────┐   instructions + tools + handoffs를 가진 LLM          │
│  │  Agent   │   에이전트는 자체 시스템 프롬프트와 도구를 보유        │
│  └─────────┘                                                       │
│                                                                     │
│  ┌─────────┐   Python 함수에 @function_tool을 붙여서 생성          │
│  │  Tool    │   JSON 스키마가 자동으로 생성됨                       │
│  └─────────┘                                                       │
│                                                                     │
│  ┌─────────┐   에이전트 간 제어를 전달하는 메커니즘                 │
│  │ Handoff  │   도구: 제어가 돌아옴 / 핸드오프: 제어가 넘어감      │
│  └─────────┘                                                       │
│                                                                     │
│  ┌─────────┐   에이전트를 실행하고 결과를 반환                      │
│  │ Runner   │   run(), run_streamed() 등 다양한 실행 방식          │
│  └─────────┘                                                       │
│                                                                     │
│  ┌─────────┐   에이전트 실행 흐름을 추적하고 시각화                 │
│  │  Trace   │   OpenAI 대시보드에서 확인 가능                      │
│  └─────────┘                                                       │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

### Agent 간 협업: 도구(Tool) vs 핸드오프(Handoff)

| 방식 | 제어 흐름 | 사용 시점 |
|------|----------|----------|
| **도구(as_tool)** | 호출 후 제어가 **돌아옴** | 다른 에이전트의 결과를 받아서 추가 처리할 때 |
| **핸드오프(Handoff)** | 제어가 다른 에이전트로 **넘어감** | 작업을 완전히 위임할 때 |

```
도구 (Tool):     A ──호출──▶ B ──결과──▶ A (제어가 A로 복귀)
핸드오프 (Handoff): A ──위임──▶ B (제어가 B로 이전)
```

---

## 2. 환경 설정

In [ ]:
# OpenAI Agents SDK 설치 (처음 한 번만 실행)
# 터미널에서: uv add openai-agents
# 또는 노트북에서:
# import sys
# !{sys.executable} -m pip install openai-agents

In [39]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import asyncio
import os

load_dotenv(override=True)

api_key = os.getenv('OPENAI_API_KEY')
if api_key:
    print("API key found.")
else:
    print("No API key was found — .env 파일에 OPENAI_API_KEY를 설정하세요.")

API key found.


---

## 3. 에이전트 워크플로우

동일한 작업을 **서로 다른 성격**의 에이전트에게 맡겨보겠습니다.

시나리오: AI 기반 고객 서비스 자동화 SaaS 회사 **"AiDesk"**의 영업 이메일을 작성합니다.

```
┌─────────────────────────────────────────────────────────────────┐
│                    3명의 영업 에이전트                          │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐         │
│  │  전문적 스타일 │  │  유머 스타일  │  │  간결 스타일  │         │
│  │  (격식체)     │  │  (친근체)     │  │  (핵심만)    │         │
│  └──────────────┘  └──────────────┘  └──────────────┘         │
│         │                 │                 │                   │
│         └─────────────────┼─────────────────┘                   │
│                           ▼                                     │
│                    동일한 입력 메시지                            │
│                    "콜드 영업 이메일 작성"                       │
└─────────────────────────────────────────────────────────────────┘
```

In [40]:
# 3가지 스타일의 영업 에이전트 정의

instructions1 = """당신은 AiDesk의 영업 담당자입니다.
AiDesk는 AI 기반 고객 서비스 자동화 SaaS 플랫폼으로, 
고객 문의 자동 응답, 티켓 분류, 감정 분석 기능을 제공합니다.
당신은 전문적이고 격식 있는 콜드 영업 이메일을 작성합니다."""

instructions2 = """당신은 AiDesk의 영업 담당자입니다.
AiDesk는 AI 기반 고객 서비스 자동화 SaaS 플랫폼으로,
고객 문의 자동 응답, 티켓 분류, 감정 분석 기능을 제공합니다.
당신은 유머러스하고 친근한 콜드 영업 이메일을 작성합니다.
받는 사람이 답장하고 싶어지는 재치 있는 문체를 사용합니다."""

instructions3 = """당신은 AiDesk의 영업 담당자입니다.
AiDesk는 AI 기반 고객 서비스 자동화 SaaS 플랫폼으로,
고객 문의 자동 응답, 티켓 분류, 감정 분석 기능을 제공합니다.
당신은 바쁜 영업 담당자로서, 간결하고 핵심만 담은 콜드 영업 이메일을 작성합니다."""

### Agent 클래스 이해하기

`Agent`는 OpenAI Agents SDK의 가장 기본적인 구성 요소입니다. **LLM + 지시사항 + 도구**를 하나로 묶은 객체입니다.

```python
Agent(
    name="에이전트 이름",           # 식별용 이름 (트레이스에서 표시됨)
    instructions="시스템 프롬프트",  # 에이전트의 역할과 행동 규칙 (str 또는 callable)
    model="gpt-4o-mini",           # 사용할 모델
    tools=[...],                   # 사용할 도구 목록 (@function_tool 또는 as_tool())
    handoffs=[...],                # 제어를 위임할 다른 에이전트 목록
    model_settings=ModelSettings(  # 모델 상세 설정 (선택)
        temperature=0.7,
        top_p=1.0,
    ),
    output_type=MyModel,           # 구조화된 출력 타입 - Pydantic 모델 (선택)
)
```

| 파라미터 | 필수 | 설명 |
|----------|------|------|
| `name` | O | 에이전트 식별 이름. 트레이스와 핸드오프에서 사용됨 |
| `instructions` | O | 시스템 프롬프트. 문자열 또는 `callable`(동적 생성) 가능 |
| `model` | - | 사용할 LLM 모델. 기본값은 SDK 설정에 따름 |
| `tools` | - | `@function_tool` 함수 또는 `agent.as_tool()` 목록 |
| `handoffs` | - | 제어를 넘길 수 있는 다른 Agent 객체 목록 |
| `model_settings` | - | temperature, top_p 등 모델 세부 설정 |
| `output_type` | - | Pydantic 모델로 구조화된 출력 강제 |

> **이전 노트북과 비교**: 직접 구현할 때는 시스템 프롬프트, 모델명, 도구 스키마를 각각 별도 변수로 관리했지만, `Agent` 클래스가 이를 **하나의 객체로 캡슐화**합니다.

In [ ]:
# Agent 객체 생성 — 이전 노트북의 시스템 프롬프트 + 모델 설정과 동일한 역할

sales_agent1 = Agent(
    name="professional_sales_agent",
    instructions=instructions1,
    model="gpt-4o-mini"
)

sales_agent2 = Agent(
    name="humorous_sales_agent",
    instructions=instructions2,
    model="gpt-4o-mini"
)

sales_agent3 = Agent(
    name="concise_sales_agent",
    instructions=instructions3,
    model="gpt-4o-mini"
)

print(f"에이전트 생성 완료: {sales_agent1.name}, {sales_agent2.name}, {sales_agent3.name}")

에이전트 생성 완료: 전문적 영업 에이전트, 유머 영업 에이전트, 간결 영업 에이전트


### 3.1 단일 에이전트 실행 (스트리밍)

In [42]:
# 스트리밍 방식으로 에이전트 실행
# Runner.run_streamed()는 토큰을 실시간으로 받아볼 수 있습니다

result = Runner.run_streamed(sales_agent1, input="콜드 영업 이메일을 작성해주세요.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

물론입니다! 다음은 AiDesk의 서비스에 대한 콜드 영업 이메일 샘플입니다.

---

제목: 고객 서비스 혁신을 위한 제안 – AiDesk의 AI 기반 솔루션

안녕하세요 [고객 이름]님,

저는 AiDesk의 영업 담당자 [당신의 이름]입니다. 귀사의 비즈니스가 더욱 효과적이고 효율적인 고객 서비스를 제공하는 데 도움을 드리고자 이렇게 연락드립니다.

우리는 고객 문의 자동 응답, 티켓 분류, 감정 분석 기능을 통해 고객 서비스 운영의 혁신을 일으킬 수 있는 AI 기반 SaaS 플랫폼을 제공합니다. 앞으로 더 많은 기업들이 데이터 기반의 의사 결정을 통해 고객 만족도를 높이고, 운영 비용을 절감하는 데 집중하고 있습니다.

AiDesk의 주요 기능은 다음과 같습니다:

1. **자동 응답 시스템**: 고객 문의에 즉시 대응하여 대기 시간을 최소화합니다.
2. **티켓 분류 기능**: 문의 내용을 자동으로 분석하고 적절한 부서에 분류하여 처리 시간을 단축합니다.
3. **감정 분석**: 고객의 감정을 실시간으로 파악하여 보다 나은 맞춤형 서비스를 제공합니다.

이러한 기능들은 귀사가 고객과의 연결을 강화하고, 서비스 품질을 개선하는 데 큰 도움이 될 것입니다. 귀사의 비즈니스 모델에 어떻게 시너지를 낼 수 있을지 구체적인 논의를 하고 싶습니다.

혹시 [날짜/시간]에 짧은 미팅을 진행할 수 있을까요? 귀사의 고객 서비스 팀이 더욱 효율적으로 운영될 수 있도록 지원할 수 있는 방법에 대해 말씀드리고 싶습니다.

감사합니다.

[당신의 이름]  
[직책]  
AiDesk  
[전화번호]  
[이메일 주소]  
[웹사이트 URL]

--- 

이 이메일을 기반으로 세부사항을 조정하여 사용하시기 바랍니다.

### 3.2 병렬 실행: 3개 에이전트 동시 실행

이전 노트북에서 학습한 **병렬화(Parallelization)** 패턴을 Agents SDK로 구현합니다.

`asyncio.gather()`를 사용하여 3개의 에이전트를 **동시에** 실행합니다.

In [43]:
message = "콜드 영업 이메일을 작성해주세요."

# trace()로 실행 흐름을 기록합니다 — OpenAI 대시보드에서 확인 가능
with trace("병렬 콜드 이메일 생성"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

outputs = [result.final_output for result in results]

labels = ["전문적 스타일", "유머 스타일", "간결 스타일"]
for label, output in zip(labels, outputs):
    print(f"{'='*60}")
    print(f"[{label}]")
    print(f"{'='*60}")
    print(output)
    print()

[전문적 스타일]
물론입니다. 아래는 AiDesk의 AI 기반 고객 서비스 자동화 SaaS 플랫폼에 관한 콜드 영업 이메일 예시입니다.

---

**Subject:** 고객 서비스 혁신의 기회를 놓치지 마세요!

안녕하세요 [고객님 성함]님,

저는 AiDesk의 영업 담당자 [당신의 이름]입니다. 바쁘신 와중에도 이 이메일을 읽어주셔서 감사합니다.

최근 고객 서비스의 중요성이 그 어느 때보다 커지고 있는 가운데, AiDesk는 AI 기반의 고객 서비스 자동화 솔루션을 통해 귀사의 고객 경험을 한층 향상시키기 위해 준비하고 있습니다. 저희 플랫폼은 다음과 같은 기능을 제공합니다:

- **고객 문의 자동 응답**: 정교한 AI 알고리즘을 통해 고객의 질문에 신속하고 정확하게 응답합니다.
- **티켓 분류**: 고객 문의를 자동으로 분류하여 보다 효율적인 처리 및 대응이 가능하게 합니다.
- **감정 분석**: 고객 의견을 분석하여 서비스 개선의 기초 자료로 활용할 수 있습니다.

이러한 특장점을 통해 고객의 만족도를 높이고 운영 효율성을 극대화할 수 있습니다. [고객사 이름] 역시 이러한 변화를 통해 더욱 우수한 고객 서비스를 제공할 수 있을 것이라 믿습니다.

혹시 자세한 정보를 원하시거나 데모를 원하신다면 편하신 시간에 미팅을 진행해보면 좋겠습니다. 아래의 링크를 통해 예약하실 수 있습니다: [미팅 예약 링크]

감사합니다. 귀하와의 대화를 기대하며, 좋은 하루 되시길 바랍니다.

최고의 경의를 표하며,

[당신의 이름]  
[당신의 직함]  
AiDesk  
[전화번호]  
[이메일 주소]  
[웹사이트 URL]  

--- 

이 이메일 템플릿을 필요에 따라 수정하여 사용하실 수 있습니다.

[유머 스타일]
제목: 안녕하세요! 고객 서비스의 슈퍼히어로가 되어보세요! 🦸‍♂️

안녕하세요 [받는 사람 이름]님!

혹시 요즘 고객 서비스 팀이 꽤 바쁘신가요? 고객 문의가 칭찬에도 불구하고 조금 지겹게 느껴진다면, 저희 AiDesk가 도움을 드릴 수 있을

### 3.3 최적 이메일 선택 (Voting 패턴)

3개의 이메일 중 **가장 효과적인 이메일**을 선택하는 에이전트를 추가합니다.

이것은 이전 노트북에서 배운 **Parallelization → Voting** 패턴의 구현입니다.

```
┌──────────────┐  ┌──────────────┐  ┌──────────────┐
│  에이전트 1   │  │  에이전트 2   │  │  에이전트 3   │
│  이메일 생성  │  │  이메일 생성  │  │  이메일 생성  │
└──────┬───────┘  └──────┬───────┘  └──────┬───────┘
       │                 │                 │
       └─────────────────┼─────────────────┘
                         ▼
              ┌──────────────────────┐
              │   선택 에이전트      │
              │  (최적 이메일 선별)   │
              └──────────┬───────────┘
                         ▼
                   최종 선택 이메일
```

In [44]:
# 이메일 선택 에이전트 — 고객 관점에서 가장 좋은 이메일을 고릅니다

sales_picker = Agent(
    name="이메일 선택 에이전트",
    instructions="""주어진 콜드 영업 이메일 후보들 중에서 가장 좋은 것을 선택하세요.
당신이 고객이라고 상상하고, 가장 답장하고 싶은 이메일을 고르세요.
설명 없이 선택한 이메일 본문만 출력하세요.""",
    model="gpt-4o-mini"
)

In [45]:
message = "콜드 영업 이메일을 작성해주세요."

with trace("최적 영업 이메일 선택"):
    # Step 1: 3개의 이메일을 병렬 생성
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    # Step 2: 선택 에이전트에게 전달
    emails = "콜드 영업 이메일 후보들:\n\n" + "\n\n---이메일---\n\n".join(outputs)
    best = await Runner.run(sales_picker, emails)

    print("최적 영업 이메일:")
    print("=" * 60)
    print(best.final_output)

최적 영업 이메일:
제목: 고객 서비스, 더 이상 '고객'이 아닌 '차렷'이 필요하신가요?

안녕하세요, [받는 분 이름]님!

한 번도 만나보지 못한 저를 소개할 수 있는 기회를 주셔서 감사합니다! 저는 AiDesk의 영업 담당자(?)로, 인공지능의 도움을 받아 고객 서비스의 효율성을 한 단계 끌어올리고 있는 팀에 속해 있습니다.

혹시 고객 문의가 쌓이면서 마치 크리스마스 트리처럼 성냥선물이 되어가고 계신가요? 😅 AiDesk는 고객 문의를 자동으로 응답하고, 티켓을 뚝딱뚝딱 분류해 주며, 감정 분석까지 할 수 있는 힘을 가지고 있습니다! 심지어 저희 플랫폼을 통해 "그냥 넘어가자!"라는 생각이 드는 고객들의 분노도 달래줄 수 있습니다. (이건 진짜예요!)

이렇게 말씀드리고 나니, “어머나, 어떻게 살아야 하지?”라는 생각이 드실 수도 있겠지만, 걱정하지 마세요. 저희와 함께라면 고객 서비스가 더 이상 머리 아픈 숙제가 아닌 마법 같은 일이 될 거랍니다!

잠시 시간을 내셔서 저희의 데모를 보시는 게 어떨까요? 확실히 우편차가 아니라 고백을 하는 기분으로 초대합니다! 😉 [귀하의 링크 삽입]

아무리 바쁘셔도 이 이메일은 빠르게 답장해 주시면 기념으로 귀여운 애니멀 GIF를 보내드릴 수도 있습니다! (무조건 귀엽습니다. 보장해 드려요!)

언제든지 질문이 있으시면 저에게 문의하시기 바랍니다. 당신의 고객 서비스가 업그레이드되는 신나는 여정을 함께하고 싶습니다!

따뜻한 하루 되세요!

[당신의 이름]  
AiDesk 팀  
[전화번호]  
[이메일 주소]  


### Trace 확인하기

`trace()`로 감싼 실행은 OpenAI 대시보드에서 시각적으로 확인할 수 있습니다:

https://platform.openai.com/traces

트레이스를 확인하면 각 에이전트가 어떤 순서로, 얼마나 걸려서 실행되었는지 확인할 수 있습니다.

---

## 4. @function_tool로 도구 정의하기

이전 노트북에서는 도구를 정의할 때 **JSON 스키마를 직접 작성**해야 했습니다.

Agents SDK에서는 `@function_tool` 데코레이터만 붙이면 **자동으로 JSON 스키마가 생성**됩니다.

```
┌─────────────────────────────────────────────────────────────────────┐
│                  도구 정의 방식 비교                                │
├────────────────────────────────┬────────────────────────────────────┤
│     이전 (직접 JSON 스키마)     │     이번 (@function_tool)         │
├────────────────────────────────┼────────────────────────────────────┤
│  1. Python 함수 작성           │  1. Python 함수에 데코레이터 추가   │
│  2. JSON 스키마 수동 작성      │     → 스키마 자동 생성!            │
│  3. handle_tool_calls() 구현   │     → 호출도 자동 처리!            │
│  4. while 루프에서 직접 관리   │     → Runner가 자동 관리!          │
│                                │                                    │
│  약 30줄                       │  약 5줄                            │
└────────────────────────────────┴────────────────────────────────────┘
```

In [46]:
# @function_tool로 도구 정의 — 이메일 전송 시뮬레이션
# 실제 이메일 전송 대신 로깅으로 대체합니다

sent_emails = []  # 전송된 이메일을 저장할 리스트

@function_tool
def send_email(to: str, subject: str, body: str) -> Dict[str, str]:
    """영업 대상에게 이메일을 전송합니다."""
    email_record = {"to": to, "subject": subject, "body": body}
    sent_emails.append(email_record)
    print(f"\n{'='*50}")
    print(f"[이메일 전송됨]")
    print(f"수신: {to}")
    print(f"제목: {subject}")
    print(f"{'='*50}")
    print(body)
    print(f"{'='*50}\n")
    return {"status": "success", "message": f"{to}에게 이메일이 전송되었습니다."}

In [ ]:
# @function_tool이 자동으로 생성한 도구 정보를 확인해봅시다
# 이전에 JSON 스키마를 직접 작성했던 것과 비교해보세요!
print(f"type(send_email): {type(send_email)}")
print(f"도구 이름: {send_email.name}")
print(f"도구 설명: {send_email.description}")
print(f"도구 파라미터 스키마: {send_email.params_json_schema}")

도구 이름: send_email
도구 설명: 영업 대상에게 이메일을 전송합니다.
도구 파라미터 스키마: {'properties': {'to': {'title': 'To', 'type': 'string'}, 'subject': {'title': 'Subject', 'type': 'string'}, 'body': {'title': 'Body', 'type': 'string'}}, 'required': ['to', 'subject', 'body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}


> **참고**: 함수의 **docstring**이 자동으로 `description`이 되고, **타입 힌트**가 자동으로 JSON 스키마의 `type`이 됩니다. 이전에 직접 작성하던 모든 것이 자동화되었습니다!

---

## 5. 에이전트를 도구로 변환: as_tool()

Agents SDK의 강력한 기능 중 하나는 **에이전트 자체를 도구로 변환**할 수 있다는 것입니다.

### 왜 에이전트를 도구로 변환하는가?

Section 3에서는 `asyncio.gather()`로 에이전트들을 **우리 코드에서 직접** 병렬 실행하고, 결과를 모아서 다시 선택 에이전트에게 전달했습니다. 이 방식은 **개발자가 흐름을 하드코딩**하는 워크플로우입니다.

에이전트를 도구로 변환하면, **상위 에이전트(오케스트레이터)가 스스로 판단하여** 하위 에이전트를 호출할 수 있습니다. 이것이 진정한 에이전트 패턴입니다.

```
┌──────────────────────────────────────────────────────────────────────┐
│  [워크플로우] Section 3 방식 — 개발자가 흐름을 코드로 제어          │
│                                                                      │
│  Python 코드:                                                        │
│    results = await asyncio.gather(                                   │
│        Runner.run(agent1, msg),    ← 개발자가 직접 호출              │
│        Runner.run(agent2, msg),                                      │
│        Runner.run(agent3, msg),                                      │
│    )                                                                 │
│    best = await Runner.run(picker, results)  ← 개발자가 직접 전달    │
├──────────────────────────────────────────────────────────────────────┤
│  [에이전트] as_tool() 방식 — LLM이 흐름을 자율적으로 결정           │
│                                                                      │
│  매니저 에이전트가 스스로:                                            │
│    1. professional_agent 도구 호출  ← LLM이 판단                     │
│    2. humorous_agent 도구 호출      ← LLM이 판단                     │
│    3. concise_agent 도구 호출       ← LLM이 판단                     │
│    4. 결과 비교 후 최적 선택        ← LLM이 판단                     │
│    5. 불만족시 다시 호출 가능       ← LLM이 판단                     │
└──────────────────────────────────────────────────────────────────────┘
```

### as_tool()의 핵심 특징

| 특징 | 설명 |
|------|------|
| **전문성 캡슐화** | 에이전트의 고유한 instructions가 유지되므로 역할 분리가 명확 |
| **오케스트레이터 패턴** | 상위 에이전트가 여러 하위 에이전트를 호출하고 결과를 종합 |
| **함수 도구와 통합** | `@function_tool` 함수와 같은 `tools` 리스트에 함께 배치 가능 |
| **동적 의사결정** | LLM이 어떤 에이전트를 호출할지, 몇 번 호출할지 스스로 결정 |

### as_tool() vs @function_tool 비교

```
@function_tool                          Agent.as_tool()
┌──────────────────────┐               ┌──────────────────────┐
│  단순한 Python 함수   │               │  Agent (LLM+지시사항) │
│  - 고정된 로직        │               │  - 자체 추론 능력     │
│  - 입력→출력 결정적   │               │  - 복잡한 텍스트 생성 │
│  - 예: DB 조회, 전송  │               │  - 예: 이메일 작성    │
└──────────────────────┘               └──────────────────────┘
         │                                       │
         └───────── 둘 다 tools 리스트에 배치 ────┘
                          │
                    ┌──────────┐
                    │ 상위     │
                    │ 에이전트 │ ← 어떤 도구든 동일하게 호출
                    └──────────┘
```

> **참고**: Section 7에서는 도구 호출과 다른 방식인 **핸드오프(Handoff)** 를 배웁니다. 도구는 결과를 받아서 추가 판단이 가능하지만, 핸드오프는 제어를 완전히 다른 에이전트로 넘깁니다.

In [48]:
# 영업 에이전트를 도구로 변환

description = "콜드 영업 이메일을 작성합니다."

tool1 = sales_agent1.as_tool(tool_name="professional_agent", tool_description="전문적이고 격식 있는 " + description)
tool2 = sales_agent2.as_tool(tool_name="humorous_agent", tool_description="유머러스하고 친근한 " + description)
tool3 = sales_agent3.as_tool(tool_name="concise_agent", tool_description="간결하고 핵심만 담은 " + description)

# 도구 목록: 3개의 에이전트 도구 + 1개의 함수 도구
tools = [tool1, tool2, tool3, send_email]

for t in tools:
    print(f"  - {t.name}: {t.description[:50]}...")

  - professional_agent: 전문적이고 격식 있는 콜드 영업 이메일을 작성합니다....
  - humorous_agent: 유머러스하고 친근한 콜드 영업 이메일을 작성합니다....
  - concise_agent: 간결하고 핵심만 담은 콜드 영업 이메일을 작성합니다....
  - send_email: 영업 대상에게 이메일을 전송합니다....


---

## 6. 세일즈 매니저: 오케스트레이터 에이전트

이제 모든 도구를 조합하여 **세일즈 매니저** 에이전트를 만듭니다.

이것은 이전 노트북에서 배운 **Orchestrator-Workers** 패턴의 실제 구현입니다!

```
┌──────────────────────────────────────────────────────────────────┐
│                      세일즈 매니저                               │
│                                                                  │
│                  ┌─────────────────┐                            │
│                  │  세일즈 매니저   │                            │
│                  │  (Orchestrator) │                            │
│                  └────────┬────────┘                            │
│                           │                                      │
│           ┌───────────────┼───────────────┐                      │
│           ▼               ▼               ▼                      │
│     ┌──────────┐   ┌──────────┐   ┌──────────┐                 │
│     │ 전문적   │   │ 유머     │   │ 간결     │  ← 에이전트     │
│     │ 에이전트 │   │ 에이전트 │   │ 에이전트 │    도구 호출     │
│     └──────────┘   └──────────┘   └──────────┘                 │
│           │               │               │                      │
│           └───────────────┼───────────────┘                      │
│                           ▼                                      │
│                    최적 이메일 선택                               │
│                           │                                      │
│                           ▼                                      │
│                    ┌──────────┐                                  │
│                    │send_email│  ← 함수 도구 호출                │
│                    └──────────┘                                  │
│                                                                  │
└──────────────────────────────────────────────────────────────────┘
```

In [ ]:
manager_instructions = """
You are a Sales Manager at AiDesk. Your goal is to find the single best cold sales email and send it.

## Your EXACT workflow (do not deviate):

Step 1: Call professional_agent, humorous_agent, and concise_agent to get 3 drafts.
Step 2: Compare the 3 drafts and pick the best one.
Step 3: Call send_email ONCE with the best draft.
Step 4: STOP. You are done.

## ABSOLUTE RULES — violation is a critical failure:

- Each of professional_agent, humorous_agent, concise_agent must be called EXACTLY ONCE. 
  Do NOT call any of them a second time for any reason.
- send_email must be called EXACTLY ONCE.
- Do NOT rewrite, improve, or regenerate any draft. Use the draft exactly as received.
- After calling send_email, do NOT call any tool. Your task is complete.
- Total tool calls allowed: 4 (3 agents + 1 send_email). No more.
"""

sales_manager = Agent(
    name="sales_manager",
    instructions=manager_instructions,
    tools=tools,
    model="gpt-4o-mini"
)

print(f"에이전트: {sales_manager.name}")
print(f"도구 수: {len(sales_manager.tools)}")
print(f"사용 가능한 도구: {[t.name for t in sales_manager.tools]}")

In [52]:
# 세일즈 매니저 실행!

sent_emails.clear()  # 이전 기록 초기화

message = "'대표님께' 로 시작하는 콜드 영업 이메일을 보내주세요. 발신자는 '김영업'입니다."

with trace("영업 매니저 파이프라인"):
    result = await Runner.run(sales_manager, message, max_turns=10)

print("\n" + "=" * 60)
print("[최종 결과]")
print("=" * 60)
print(result.final_output)


[이메일 전송됨]
수신: recipient@example.com
제목: 귀사의 성장을 지원하는 AiDesk의 AI 기반 고객 서비스 솔루션 소개
대표님께,

안녕하세요. AiDesk의 김영업입니다. 귀사의 뛰어난 성과와 미래 비전에 큰 감명을 받았습니다. 저희 AiDesk는 고객 서비스 자동화를 통해 기업의 생산성 향상과 비용 절감을 지원하는 AI 기반 SaaS 플랫폼입니다.

저희 솔루션은 고객 문의 자동 응답, 티켓 분류, 그리고 감정 분석 기능을 통해 귀사의 고객 경험을 향상시키고, 운영 효율성을 극대화할 수 있습니다. 이러한 기능을 통해 귀사가 더욱 더 성장할 수 있도록 지원하고 싶습니다.

혹시 저희 솔루션에 대한 추가 정보나 상담이 필요하시다면 언제든지 편하게 연락 주시면 감사하겠습니다. 귀사의 발전에 함께할 수 있기를 기대합니다.

감사합니다.

김영업  
AiDesk 드림  
[연락처]  
[웹사이트]


[최종 결과]
이메일이 성공적으로 전송되었습니다. 내용은 다음과 같습니다:

---

**제목:** 귀사의 성장을 지원하는 AiDesk의 AI 기반 고객 서비스 솔루션 소개  

대표님께,

안녕하세요. AiDesk의 김영업입니다. 귀사의 뛰어난 성과와 미래 비전에 큰 감명을 받았습니다. 저희 AiDesk는 고객 서비스 자동화를 통해 기업의 생산성 향상과 비용 절감을 지원하는 AI 기반 SaaS 플랫폼입니다.

저희 솔루션은 고객 문의 자동 응답, 티켓 분류, 그리고 감정 분석 기능을 통해 귀사의 고객 경험을 향상시키고, 운영 효율성을 극대화할 수 있습니다. 이러한 기능을 통해 귀사가 더욱 더 성장할 수 있도록 지원하고 싶습니다.

혹시 저희 솔루션에 대한 추가 정보나 상담이 필요하시다면 언제든지 편하게 연락 주시면 감사하겠습니다. 귀사의 발전에 함께할 수 있기를 기대합니다.

감사합니다.

김영업  
AiDesk 드림  
[연락처]  
[웹사이트]

---

추가로 도움이 필요하시면 언제든지 말씀해 주세요!


> **트레이스 확인**: https://platform.openai.com/traces 에서 "영업 매니저 파이프라인" 트레이스를 찾아보세요.
> 세일즈 매니저가 3개의 에이전트 도구를 호출하고, 평가 후 send_email을 호출하는 전체 흐름을 시각적으로 확인할 수 있습니다.

---

## 7. Handoff: 에이전트 간 제어 전달

**도구(Tool)** 는 호출 후 제어가 원래 에이전트로 돌아오지만, **핸드오프(Handoff)** 는 제어를 완전히 다른 에이전트로 넘깁니다.

이번에는 핸드오프를 활용하여 더 정교한 파이프라인을 만들어봅시다.

```
┌──────────────────────────────────────────────────────────────────┐
│                  도구 vs 핸드오프 비교                           │
├──────────────────────────────────────────────────────────────────┤
│                                                                  │
│  [도구] 세일즈 매니저 ──호출──▶ 영업 에이전트 ──결과──▶ 매니저  │
│         (제어 유지)      ↑                               │      │
│                          └───────────────────────────────┘      │
│                                                                  │
│  [핸드오프] 세일즈 매니저 ──위임──▶ 이메일 매니저               │
│             (제어 종료)              (제어 인수)                  │
│                                      ├──▶ 제목 작성             │
│                                      ├──▶ HTML 변환             │
│                                      └──▶ 이메일 전송           │
│                                                                  │
└──────────────────────────────────────────────────────────────────┘
```

In [53]:
# 이메일 전송 파이프라인의 보조 에이전트들을 정의합니다

# 제목 작성 에이전트 (도구로 사용)
subject_writer = Agent(
    name="이메일 제목 작성기",
    instructions="""콜드 영업 이메일의 제목을 작성합니다.
이메일 본문을 받으면 열어보고 싶은 매력적인 제목을 작성하세요.
제목만 출력하세요.""",
    model="gpt-4o-mini"
)
subject_tool = subject_writer.as_tool(
    tool_name="subject_writer",
    tool_description="콜드 영업 이메일의 제목을 작성합니다."
)

# HTML 변환 에이전트 (도구로 사용)
html_converter = Agent(
    name="HTML 이메일 변환기",
    instructions="""텍스트 이메일 본문을 HTML 이메일로 변환합니다.
마크다운이 포함된 텍스트 이메일을 받으면 깔끔하고 전문적인 HTML 이메일로 변환하세요.
인라인 CSS를 사용하여 보기 좋게 디자인하세요.""",
    model="gpt-4o-mini"
)
html_tool = html_converter.as_tool(
    tool_name="html_converter",
    tool_description="텍스트 이메일 본문을 HTML 이메일로 변환합니다."
)

In [54]:
# HTML 이메일 전송 도구

@function_tool
def send_html_email(to: str, subject: str, html_body: str) -> Dict[str, str]:
    """제목과 HTML 본문으로 이메일을 전송합니다."""
    email_record = {"to": to, "subject": subject, "html_body": html_body}
    sent_emails.append(email_record)
    print(f"\n{'='*50}")
    print(f"[HTML 이메일 전송됨]")
    print(f"수신: {to}")
    print(f"제목: {subject}")
    print(f"{'='*50}")
    print(html_body[:500] + "..." if len(html_body) > 500 else html_body)
    print(f"{'='*50}\n")
    return {"status": "success", "message": f"{to}에게 HTML 이메일이 전송되었습니다."}

### 왜 이메일 매니저를 핸드오프로 만드는가?

Section 6에서는 `send_email`을 세일즈 매니저의 도구로 넣어서 **매니저가 직접 전송**했습니다. 이번에는 전송 프로세스를 별도 에이전트에게 **핸드오프**합니다. 왜 이렇게 바꿀까요?

#### 1. 세일즈 매니저가 결과를 돌려받을 필요가 없다

세일즈 매니저의 역할은 **최적 이메일을 선택**하는 것까지입니다. 전송 결과를 받아서 추가 판단할 일이 없습니다. 이런 경우 제어를 완전히 넘기는 **핸드오프**가 적합합니다.

```
도구를 쓸 때:  매니저 → 전송 → 결과 돌아옴 → 매니저가 또 뭔가 해야 함
핸드오프:      매니저 → 전송 에이전트에게 위임 → 매니저는 끝!  ← 더 자연스러움
```

#### 2. 관심사 분리 (Separation of Concerns)

전송 프로세스에는 **제목 작성 → HTML 변환 → 전송**이라는 자체 파이프라인이 있습니다. 이것을 매니저의 도구로 넣으면 매니저가 6개 도구(영업 에이전트 3개 + 제목 + HTML + 전송)를 관리해야 합니다.

```
┌──────────────────────────────────────────────────────────────────┐
│  [도구 방식] 매니저가 모든 것을 직접 관리                         │
│                                                                  │
│  매니저의 tools: [agent1, agent2, agent3,                        │
│                  subject_writer, html_converter, send_html_email]│
│                                                                  │
│  → 도구가 6개로 복잡해짐                                         │
│  → 매니저 instructions에 전송 절차까지 설명해야 함               │
│  → 역할이 "이메일 선택 + 포맷팅 + 전송"으로 비대해짐             │
├──────────────────────────────────────────────────────────────────┤
│  [핸드오프 방식] 각자 역할에 집중                                │
│                                                                  │
│  매니저의 tools: [agent1, agent2, agent3]                        │
│  매니저의 handoffs: [email_manager]                              │
│                                                                  │
│  → 매니저는 "선택"에만 집중                                      │
│  → 이메일 매니저는 "포맷팅 + 전송"에만 집중                      │
│  → 각 에이전트의 instructions가 단순하고 명확                    │
└──────────────────────────────────────────────────────────────────┘
```

#### 3. 실무에서의 핸드오프 활용 예시

| 시나리오 | 핸드오프 대상 | 이유 |
|----------|-------------|------|
| 고객 문의 → 전문 상담 | 기술지원 에이전트 | 분류 후 제어를 넘기면 됨 |
| 주문 접수 → 결제 처리 | 결제 에이전트 | 주문 에이전트가 결제 결과를 처리할 필요 없음 |
| 콘텐츠 작성 → 퍼블리싱 | 배포 에이전트 | 작성자가 배포 세부사항을 알 필요 없음 |
| **이메일 선택 → 전송** | **이메일 매니저** | **매니저가 전송 세부사항을 알 필요 없음** |

> **핵심 판단 기준**: 결과를 돌려받아서 **추가 판단이 필요한가?** Yes → `as_tool()`, No → `handoff`

#### 주의: 핸드오프 에이전트의 name은 영문으로!

SDK는 핸드오프 에이전트의 `name`으로 `transfer_to_{name}` 형태의 tool name을 자동 생성합니다. **한글 name을 사용하면** `transfer_to________`로 변환되어 LLM이 핸드오프를 인식하지 못합니다.

```python
# ❌ 한글 name → transfer_to________  (LLM이 인식 불가)
Agent(name="이메일 매니저", ...)

# ✅ 영문 name → transfer_to_email_manager  (정상 동작)
Agent(name="email_manager", ...)
```

> **팁**: `name`은 영문으로, `instructions`는 한국어로 작성하세요. `name`은 내부 식별자이고 `instructions`가 에이전트의 실제 행동을 결정합니다.

In [55]:
# 이메일 매니저 에이전트 — 핸드오프 대상
# 이메일 본문을 받아서 제목 작성 → HTML 변환 → 전송까지 처리합니다
#
# 주의: handoff 대상 에이전트의 name은 반드시 영문으로!
# SDK가 name으로 "transfer_to_{name}" 형태의 tool name을 자동 생성하는데,
# 한글이 포함되면 "transfer_to________"로 변환되어 LLM이 인식하지 못합니다.

emailer_agent = Agent(
    name="email_manager",
    instructions="""당신은 이메일 포맷팅 및 전송을 담당합니다.
이메일 본문을 받으면 다음 순서로 처리하세요:
1. subject_writer 도구로 이메일 제목을 작성
2. html_converter 도구로 본문을 HTML로 변환
3. send_html_email 도구로 이메일을 전송
4. subject_writer, html_converter, send_html_email 도구는 각각 한 번씩만 호출하세요. 절대 같은 도구를 여러 번 호출하지 마세요.

수신자 주소가 명시되지 않은 경우 'prospect@example.com'을 사용하세요.""",
    tools=[subject_tool, html_tool, send_html_email],
    model="gpt-4o-mini",
    handoff_description="Format the email as HTML and send it."
)

In [56]:
# 확인: 도구와 핸드오프 목록

agent_tools = [tool1, tool2, tool3]  # 에이전트 도구 (이메일 생성)
handoffs = [emailer_agent]            # 핸드오프 대상 (이메일 포맷팅+전송)

print("[에이전트 도구 — 호출 후 제어 복귀]")
for t in agent_tools:
    print(f"  - {t.name}")

print("\n[핸드오프 — 제어 전달]")
for h in handoffs:
    print(f"  - {h.name}: {h.handoff_description}")

[에이전트 도구 — 호출 후 제어 복귀]
  - professional_agent
  - humorous_agent
  - concise_agent

[핸드오프 — 제어 전달]
  - email_manager: Format the email as HTML and send it.


---

## 8. 완전한 파이프라인: 세일즈 매니저 + 핸드오프

이제 **도구**와 **핸드오프**를 결합한 완전한 영업 자동화 파이프라인을 구축합니다.

```
┌──────────────────────────────────────────────────────────────────────┐
│                    완전한 영업 자동화 파이프라인                      │
│                                                                      │
│  ┌────────────────────────────────────────────────┐                  │
│  │              세일즈 매니저                       │                  │
│  │                                                 │                  │
│  │  1. 에이전트 도구 호출 (제어 복귀)               │                  │
│  │     ├─ professional_agent → 이메일 A             │                  │
│  │     ├─ humorous_agent → 이메일 B                 │                  │
│  │     └─ concise_agent → 이메일 C                  │                  │
│  │                                                 │                  │
│  │  2. 최적 이메일 선택                             │                  │
│  │                                                 │                  │
│  │  3. 핸드오프 (제어 전달) ─────────────────────┐  │                  │
│  └────────────────────────────────────────────── │──┘                  │
│                                                  │                    │
│                                                  ▼                    │
│  ┌────────────────────────────────────────────────┐                  │
│  │              이메일 매니저                       │                  │
│  │                                                 │                  │
│  │  4. subject_writer → 제목 생성                  │                  │
│  │  5. html_converter → HTML 변환                  │                  │
│  │  6. send_html_email → 이메일 전송               │                  │
│  └────────────────────────────────────────────────┘                  │
│                                                                      │
└──────────────────────────────────────────────────────────────────────┘
```

### 워크플로우 → 에이전트 전환의 핵심

이전 노트북에서 배운 Anthropic의 정의를 기억하시나요?

- **워크플로우**: 미리 정의된 코드 경로로 실행
- **에이전트**: LLM이 자체적으로 프로세스와 도구 사용을 결정

이 파이프라인에서 **세일즈 매니저가 어떤 이메일을 선택할지는 LLM이 결정**합니다.
결과에 만족하지 않으면 도구를 다시 호출할 수도 있습니다.
이것이 단순한 워크플로우가 아닌 **에이전트**인 이유입니다.

In [61]:
# 완전한 세일즈 매니저 에이전트 (도구 + 핸드오프)
from agents import ModelSettings

full_manager_instructions = """
You are a Sales Manager at AiDesk. Your goal is to find the single best cold sales email and send it.

Follow these steps carefully:

1. Call each of the three tools ONCE: professional_agent, humorous_agent, concise_agent.
   Each generates a Korean cold sales email draft. Call all three tools in a single turn .
   Never Never call any sales_agent(professional_agent, humorous_agent, concise_agent) tool more than once.

2. After receiving all three drafts, pick the single best one. Do NOT call any sales_agent tools again.

3. Immediately hand off the winning email text to email_manager using transfer_to_email_manager.

Rules:
- Call each sales_agent tool exactly once.
- Never regenerate or rewrite drafts.
- Always finish by handing off to email_manager.
"""

# parallel_tool_calls=False: gpt-4o-mini가 한 턴에 handoff를 여러 번 호출하는 것을 방지
# (Multiple handoffs requested 에러 방지)
sales_manager_v2 = Agent(
    name="sales_manager_v2",
    instructions=full_manager_instructions,
    tools=agent_tools,       # 에이전트 도구 (이메일 생성)
    handoffs=handoffs,       # 핸드오프 (이메일 포맷팅+전송)
    model="gpt-4o-mini",
    model_settings=ModelSettings(parallel_tool_calls=False)
)

print(f"에이전트: {sales_manager_v2.name}")
print(f"도구: {[t.name for t in sales_manager_v2.tools]}")
print(f"핸드오프: {[h.name for h in sales_manager_v2.handoffs]}")

에이전트: sales_manager_v2
도구: ['professional_agent', 'humorous_agent', 'concise_agent']
핸드오프: ['email_manager']


In [62]:
# 완전한 파이프라인 실행!
# max_turns: 에이전트의 최대 실행 턴 수를 제한하여 무한 루프 방지

sent_emails.clear()  # 이전 기록 초기화

message = "'CTO님께' 로 시작하는 콜드 영업 이메일을 보내주세요. 발신자는 '박서연'입니다."

with trace("자동화 영업 파이프라인"):
    result = await Runner.run(sales_manager_v2, message, max_turns=10)

print("\n" + "=" * 60)
print("[파이프라인 실행 완료]")
print("=" * 60)
print(f"최종 에이전트: {result.last_agent.name}")
print(f"최종 출력:\n{result.final_output}")


[HTML 이메일 전송됨]
수신: prospect@example.com
제목: 귀사의 업무 효율성을 위한 AI 솔루션 제안
<p>CTO님께,</p><p>안녕하세요. AiDesk의 박서연입니다. AI 기술 분야에서의 혁신을 선도하며, 기업들이 효율성을 극대화하고 경쟁력을 강화하는 데 기여하고 있습니다.</p><p>저희 AiDesk 플랫폼은 고객 문의 자동 응답, 티켓 분류, 그리고 감정 분석 기능을 통해 고객 서비스의 품질을 크게 향상시키는 것을 목표로 하고 있습니다. 귀사의 프로젝트와 관련하여 저희 솔루션이 매우 유용할 것이라 생각하여 간단히 연락드리게 되었습니다.</p><p>귀하의 업무 효율성을 높이는 데 도움이 될 수 있는 방안에 대해 직접 말씀드릴 기회를 요청드립니다. 가능하신 시간에 짧은 미팅을 통해 자세히 설명드리고 싶습니다.</p><p>긍정적인 검토 부탁드립니다.</p><p>감사합니다.</p><p>박서연 드림<br>AiDesk<br>[연락처]<br>[웹사이트]</p>


[파이프라인 실행 완료]
최종 에이전트: email_manager
최종 출력:
이메일이 성공적으로 **prospect@example.com**에게 전송되었습니다. 내용은 다음과 같습니다:

---

**제목:** 귀사의 업무 효율성을 위한 AI 솔루션 제안

**메일 본문:**
CTO님께,

안녕하세요. AiDesk의 박서연입니다. AI 기술 분야에서의 혁신을 선도하며, 기업들이 효율성을 극대화하고 경쟁력을 강화하는 데 기여하고 있습니다.

저희 AiDesk 플랫폼은 고객 문의 자동 응답, 티켓 분류, 그리고 감정 분석 기능을 통해 고객 서비스의 품질을 크게 향상시키는 것을 목표로 하고 있습니다. 귀사의 프로젝트와 관련하여 저희 솔루션이 매우 유용할 것이라 생각하여 간단히 연락드리게 되었습니다.

귀하의 업무 효율성을 높이는 데 도움이 될 수 있는 방안에 대해 직접 말씀드릴 기회를 요청드립니다. 가능하신 시간에 짧은 미팅을 통해 자세히 설명드리고 싶습니다.

긍정적인 검토 부탁드립

### Trace 확인

https://platform.openai.com/traces 에서 **"자동화 영업 파이프라인"** 트레이스를 확인해보세요.

다음 흐름을 시각적으로 볼 수 있습니다:
1. 영업 매니저가 3개의 에이전트 도구 호출
2. 최적 이메일 선택
3. 이메일 매니저에게 핸드오프
4. 이메일 매니저가 제목 작성, HTML 변환, 전송 순서로 처리

---

## 9. 전송된 이메일 확인

In [63]:
# 전송된 이메일 기록 확인

print(f"총 전송된 이메일 수: {len(sent_emails)}\n")

for i, email in enumerate(sent_emails, 1):
    print(f"--- 이메일 #{i} ---")
    print(f"수신: {email.get('to', 'N/A')}")
    print(f"제목: {email.get('subject', 'N/A')}")
    if 'html_body' in email:
        print(f"형식: HTML")
        print(f"본문 미리보기: {email['html_body'][:200]}...")
    else:
        print(f"형식: 텍스트")
        print(f"본문: {email.get('body', 'N/A')[:200]}...")
    print()

총 전송된 이메일 수: 1

--- 이메일 #1 ---
수신: prospect@example.com
제목: 귀사의 업무 효율성을 위한 AI 솔루션 제안
형식: HTML
본문 미리보기: <p>CTO님께,</p><p>안녕하세요. AiDesk의 박서연입니다. AI 기술 분야에서의 혁신을 선도하며, 기업들이 효율성을 극대화하고 경쟁력을 강화하는 데 기여하고 있습니다.</p><p>저희 AiDesk 플랫폼은 고객 문의 자동 응답, 티켓 분류, 그리고 감정 분석 기능을 통해 고객 서비스의 품질을 크게 향상시키는 것을 목표로 하고 있습니다. 귀사의 프...



In [64]:

# HTML 이메일이 있다면 Jupyter에서 렌더링해봅시다
# iframe을 사용하여 HTML의 CSS가 노트북 레이아웃에 영향을 주지 않도록 합니다

from IPython.display import HTML, display
import html as html_module

html_emails = [e for e in sent_emails if 'html_body' in e]
if html_emails:
    print(f"HTML 이메일 {len(html_emails)}개를 렌더링합니다:\n")
    for email in html_emails:
        print(f"제목: {email['subject']}")
        escaped = html_module.escape(email['html_body'])
        iframe_html = f'''<iframe srcdoc="{escaped}"
            style="width:100%; height:600px; border:1px solid #ccc; border-radius:4px;"
            sandbox="allow-same-origin"></iframe>'''
        display(HTML(iframe_html))
else:
    print("HTML 이메일이 없습니다.")


HTML 이메일 1개를 렌더링합니다:

제목: 귀사의 업무 효율성을 위한 AI 솔루션 제안


/Users/windfree/workspace/ws.study/ai-engineering/.venv/lib/python3.13/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


---

## 10. Guardrails: 에이전트 입력 보호

**Guardrails**는 에이전트가 실행되기 **전/후에** 입력을 검사하여, 부적절하거나 위험한 요청을 차단하는 보호 메커니즘입니다.

### 왜 Guardrails가 필요한가?

이전 섹션에서 우리는 **도구(Tool)** 와 **핸드오프(Handoff)** 로 에이전트의 **능력**을 확장했습니다. 하지만 능력이 강해질수록 **보호 장치**도 필요합니다. 예를 들어:

- 개인정보(이름, 이메일)가 포함된 요청 차단
- 유해하거나 부적절한 콘텐츠 필터링
- 회사 컴플라이언스 정책 위반 감지

### input_guardrail vs output_guardrail

| 구분 | input_guardrail | output_guardrail |
|------|----------------|-----------------|
| **검사 시점** | 에이전트 실행 **전** | 에이전트 실행 **후** |
| **검사 대상** | 사용자 입력 메시지 | 에이전트의 최종 출력 |
| **예외** | `InputGuardrailTripwireTriggered` | `OutputGuardrailTripwireTriggered` |
| **사용 예시** | PII 감지, 유해 콘텐츠 차단 | 환각 검증, 톤 검사 |

### 동작 흐름

```
┌─────────────────────────────────────────────────────────────────┐
│                   Input Guardrail 동작 흐름                     │
│                                                                 │
│  사용자 입력                                                    │
│      │                                                          │
│      ▼                                                          │
│  ┌──────────────────┐                                          │
│  │  Guardrail 검사   │                                          │
│  │  (별도 에이전트)   │                                          │
│  └────────┬─────────┘                                          │
│           │                                                     │
│     ┌─────┴─────┐                                              │
│     │           │                                               │
│   통과        차단                                              │
│     │           │                                               │
│     ▼           ▼                                               │
│  에이전트    InputGuardrailTripwireTriggered                    │
│  정상 실행   예외 발생 → 실행 중단                               │
│     │                                                           │
│     ▼                                                           │
│  최종 출력                                                      │
└─────────────────────────────────────────────────────────────────┘
```

> **이전 섹션과의 관계**: Section 4-8에서 도구와 핸드오프로 에이전트의 **기능**을 확장했다면, Guardrails는 에이전트의 **안전성**을 강화합니다. 실무에서는 기능과 보호를 함께 설계합니다.

### Guardrail 구현 3단계

Guardrail을 구현하려면 다음 3단계를 거칩니다:

```
┌─────────────────────────────────────────────────────────────────┐
│                  Guardrail 구현 단계                            │
│                                                                 │
│  Step 1: Guardrail 판단 에이전트 정의                          │
│  ├─ Pydantic 모델로 구조화된 출력 (output_type) 정의           │
│  └─ 에이전트가 입력을 분석하고 위반 여부를 판단                 │
│                                                                 │
│  Step 2: @input_guardrail 데코레이터로 검사 함수 정의          │
│  ├─ 판단 에이전트를 실행하고 결과를 GuardrailFunctionOutput    │
│  │  으로 반환                                                   │
│  └─ tripwire_triggered=True이면 SDK가 예외를 발생시킴          │
│                                                                 │
│  Step 3: 에이전트에 input_guardrails 파라미터로 적용            │
│  └─ Agent(..., input_guardrails=[name_guardrail])               │
└─────────────────────────────────────────────────────────────────┘
```

- **Step 1**의 `output_type`은 Section 7의 `Agent` 파라미터 표에서 소개한 **구조화된 출력**을 활용합니다. Pydantic 모델을 지정하면 에이전트가 반드시 해당 스키마에 맞는 JSON을 출력합니다.
- **Step 2**의 `@input_guardrail`은 `@function_tool`처럼 SDK가 제공하는 데코레이터입니다.
- **Step 3**에서 기존 에이전트에 guardrail을 추가하기만 하면 됩니다 — 에이전트의 나머지 설정은 변경 불필요!

In [ ]:
# Step 1: Guardrail에 필요한 추가 import
from agents import input_guardrail, GuardrailFunctionOutput, InputGuardrailTripwireTriggered
from pydantic import BaseModel

# Guardrail 판단 결과를 담을 Pydantic 모델
# output_type으로 지정하면 에이전트가 반드시 이 스키마에 맞는 JSON을 출력합니다
class NameCheckOutput(BaseModel):
    is_name_in_message: bool  # 개인 이름이 포함되어 있는지 여부
    name: str                  # 감지된 이름 (없으면 빈 문자열)

# Guardrail 판단 에이전트 — 입력에 개인 이름이 포함되어 있는지 검사
# output_type=NameCheckOutput으로 구조화된 출력을 강제합니다
guardrail_agent = Agent(
    name="Name check guardrail",
    instructions="Check if the user is including someone's personal name in what they want you to do.",
    output_type=NameCheckOutput,
    model="gpt-4o-mini"
)

print(f"Guardrail 에이전트: {guardrail_agent.name}")
print(f"출력 타입: {guardrail_agent.output_type}")

### @input_guardrail 데코레이터 이해하기

`@input_guardrail`로 장식된 함수는 다음 시그니처를 가집니다:

```python
@input_guardrail
async def my_guardrail(ctx, agent, message):
    # ctx: RunContextWrapper — 실행 컨텍스트 (context 데이터 접근 가능)
    # agent: Agent — guardrail이 적용된 에이전트 객체
    # message: str — 사용자의 입력 메시지
    
    return GuardrailFunctionOutput(
        output_info=...,          # 디버깅/로깅용 추가 정보 (dict, str 등)
        tripwire_triggered=...    # True이면 SDK가 예외를 발생시킴
    )
```

| 파라미터 | 설명 |
|----------|------|
| `output_info` | guardrail 결과의 상세 정보. 트레이스에 기록됨 |
| `tripwire_triggered` | `True` → `InputGuardrailTripwireTriggered` 예외 발생, 에이전트 실행 중단 |

> **핵심**: `tripwire_triggered=True`이면 SDK가 **자동으로** `InputGuardrailTripwireTriggered` 예외를 발생시킵니다. 개발자는 `try/except`로 이 예외를 처리하면 됩니다.

In [ ]:
# Step 2: @input_guardrail 데코레이터로 검사 함수 정의
# guardrail_agent를 실행하여 이름 포함 여부를 판단하고,
# 이름이 포함되어 있으면 tripwire_triggered=True로 차단합니다

@input_guardrail
async def name_guardrail(ctx, agent, message):
    # guardrail_agent가 메시지를 분석하여 NameCheckOutput을 반환
    result = await Runner.run(guardrail_agent, message, context=ctx.context)
    final = result.final_output  # NameCheckOutput 타입

    return GuardrailFunctionOutput(
        # output_info: 디버깅/로깅용 — 트레이스에서 확인 가능
        output_info={"found_name": final.name, "is_name": final.is_name_in_message},
        # tripwire_triggered: True이면 InputGuardrailTripwireTriggered 예외 발생
        tripwire_triggered=final.is_name_in_message
    )

print(f"Guardrail 함수 정의 완료: {name_guardrail.guardrail_function.__name__}")

In [ ]:
# Step 3: Guardrail이 적용된 세일즈 매니저 생성
# 기존 sales_manager_v2와 동일하지만 input_guardrails만 추가!
# full_manager_instructions, agent_tools, handoffs는 Section 8에서 정의한 것을 재사용합니다

guarded_sales_manager = Agent(
    name="guarded_sales_manager",
    instructions=full_manager_instructions,
    tools=agent_tools,
    handoffs=handoffs,
    model="gpt-4o-mini",
    input_guardrails=[name_guardrail]  # ← 이것만 추가!
)

print(f"에이전트: {guarded_sales_manager.name}")
print(f"도구: {[t.name for t in guarded_sales_manager.tools]}")
print(f"핸드오프: {[h.name for h in guarded_sales_manager.handoffs]}")
print(f"입력 가드레일: {[g.guardrail_function.__name__ for g in guarded_sales_manager.input_guardrails]}")

### Guardrail 테스트

이제 두 가지 케이스로 guardrail이 정상 동작하는지 확인합니다:

| 케이스 | 입력 메시지 | 예상 결과 |
|--------|-----------|----------|
| **Case 1** | 발신자가 **'박서연'** (개인 이름) | Guardrail 발동 → `InputGuardrailTripwireTriggered` 예외 |
| **Case 2** | 발신자가 **'사업개발팀'** (부서명) | Guardrail 통과 → 정상 실행 |

In [ ]:
# Case 1: 개인 이름 포함 → guardrail이 차단해야 함
# '박서연'이라는 개인 이름이 포함되어 있으므로 InputGuardrailTripwireTriggered 예외가 발생합니다

sent_emails.clear()

try:
    message = "'CTO님께' 로 시작하는 콜드 영업 이메일을 보내주세요. 발신자는 '박서연'입니다."
    with trace("Guardrail 테스트 - 이름 포함"):
        result = await Runner.run(guarded_sales_manager, message, max_turns=10)
    print(result.final_output)
except InputGuardrailTripwireTriggered as e:
    print(f"✓ Guardrail 발동! 개인 이름이 감지되어 실행이 차단되었습니다.")
    print(f"  상세 정보: {e}")

In [ ]:
# Case 2: 개인 이름 미포함 → 정상 실행되어야 함
# '사업개발팀'은 부서명이므로 guardrail을 통과하고 에이전트가 정상 실행됩니다

sent_emails.clear()

try:
    message = "'CTO님께' 로 시작하는 콜드 영업 이메일을 보내주세요. 발신자는 '사업개발팀'입니다."
    with trace("Guardrail 테스트 - 이름 미포함"):
        result = await Runner.run(guarded_sales_manager, message, max_turns=10)
    print("\n" + "=" * 60)
    print("[Guardrail 통과 — 정상 실행 완료]")
    print("=" * 60)
    print(f"최종 에이전트: {result.last_agent.name}")
    print(f"최종 출력:\n{result.final_output}")
except InputGuardrailTripwireTriggered as e:
    print(f"Guardrail 발동! 개인 이름이 감지되어 실행이 차단되었습니다.")
    print(f"상세 정보: {e}")

### Guardrails 정리

#### input_guardrail vs output_guardrail 비교

```
┌─────────────────────────────────────────────────────────────────┐
│               Guardrail 유형 비교                               │
├──────────────────────────┬──────────────────────────────────────┤
│   input_guardrail        │   output_guardrail                  │
├──────────────────────────┼──────────────────────────────────────┤
│  에이전트 실행 전 검사    │  에이전트 실행 후 검사               │
│  사용자 입력을 검증       │  에이전트 출력을 검증                │
│  @input_guardrail        │  @output_guardrail                  │
│  InputGuardrailTrip-     │  OutputGuardrailTrip-               │
│    wireTriggered 예외    │    wireTriggered 예외               │
│  PII, 유해 입력 차단     │  환각, 부적절한 응답 차단            │
└──────────────────────────┴──────────────────────────────────────┘
```

#### 실무 활용 예시

| Guardrail 유형 | 활용 시나리오 | 설명 |
|---------------|-------------|------|
| **PII 필터링** | input | 이름, 이메일, 전화번호 등 개인정보 감지 및 차단 |
| **유해 콘텐츠 차단** | input | 욕설, 혐오 표현, 불법 요청 등 필터링 |
| **컴플라이언스 검증** | input/output | 금융 규정, 의료 정보 등 규제 준수 여부 확인 |
| **환각 검증** | output | 에이전트 출력에 사실과 다른 내용이 포함되었는지 검사 |
| **톤/스타일 검사** | output | 브랜드 가이드라인에 맞는 문체인지 확인 |
| **토픽 제한** | input | 에이전트의 역할 범위를 벗어나는 요청 차단 |

---

## 11. 에이전트 디자인 패턴 정리

이번 노트북에서 사용한 패턴들을 정리해봅시다:

### 사용된 디자인 패턴

| 패턴 | 위치 | 설명 |
|------|------|------|
| **Parallelization (병렬화)** | Section 3.2 | 3개의 에이전트를 `asyncio.gather()`로 병렬 실행 |
| **Voting (투표)** | Section 3.3 | 여러 결과 중 최적 선택 |
| **Orchestrator-Workers** | Section 6 | 매니저가 하위 에이전트에게 작업 분배 |
| **Prompt Chaining** | Section 7-8 | 이메일 매니저의 순차 처리 (제목→HTML→전송) |
| **Tool Use** | Section 4-6 | `@function_tool`과 `as_tool()` |
| **Guardrails** | Section 10 | `@input_guardrail`로 에이전트 입력 보호 |

### 워크플로우 vs 에이전트

```
┌─────────────────────────────────────────────────────────────────────┐
│  이 시스템이 "에이전트"인 이유:                                     │
│                                                                     │
│  세일즈 매니저가 결과에 만족하지 않으면 도구를 다시 호출할 수 있음   │
│  → LLM이 프로세스의 진행을 동적으로 결정                            │
│  → 단순한 고정 경로 워크플로우가 아님                               │
│                                                                     │
│  이메일 매니저는 "워크플로우"에 가까움:                              │
│  → 제목 작성 → HTML 변환 → 전송이라는 고정된 순서를 따름            │
│                                                                     │
│  Guardrails는 에이전트/워크플로우 모두에 적용 가능:                  │
│  → 실행 전 입력 검증으로 안전성 확보                                │
│                                                                     │
│  실제 시스템은 워크플로우와 에이전트를 적절히 조합합니다              │
└─────────────────────────────────────────────────────────────────────┘
```

---

## 12. 연습 과제 및 확장 아이디어

### 연습 과제

1. **에이전트 추가**: 새로운 스타일의 영업 에이전트를 추가해보세요 (예: 데이터 중심, 스토리텔링 등)
2. **평가 기준 강화**: `sales_picker`에 구체적인 평가 기준(수신자 관점, CTA 명확성 등)을 추가해보세요
3. **실제 이메일 연동**: SendGrid나 Resend를 연동하여 실제 이메일을 발송해보세요
4. **Output Guardrail 추가**: `@output_guardrail`을 구현하여 생성된 이메일에 경쟁사 비하 표현이 없는지 검증해보세요
5. **다중 Guardrail**: 여러 guardrail을 동시에 적용해보세요 (PII 필터링 + 유해 콘텐츠 차단 + 토픽 제한)

### 확장 아이디어

| 아이디어 | 설명 |
|----------|------|
| 수신자 조사 에이전트 | 웹 검색 도구로 수신자 정보를 조사하고 맞춤 이메일 작성 |
| A/B 테스트 시스템 | 여러 버전의 이메일 성과를 추적하고 최적화 |
| 팔로업 에이전트 | 응답이 없을 때 자동으로 후속 이메일 작성 |
| CRM 연동 | 고객 정보 DB와 연동하여 개인화된 이메일 작성 |
| Output Guardrail | 에이전트 출력의 환각/부적절 내용을 자동 검증 |
| 다중 Guardrail 파이프라인 | PII + 유해 콘텐츠 + 컴플라이언스 등 여러 guardrail 동시 적용 |

---

## 참고 자료

- [OpenAI Agents SDK 문서](https://openai.github.io/openai-agents-python/)
- [OpenAI Agents SDK GitHub](https://github.com/openai/openai-agents-python)
- [OpenAI Traces 대시보드](https://platform.openai.com/traces)
- [Google Agent Development Kit (ADK)](https://google.github.io/adk-docs/) — 유사한 패턴, 다른 프레임워크

### 참고: Google ADK 코드 예시

Google의 ADK도 OpenAI Agents SDK와 매우 유사한 패턴을 따릅니다:

```python
# Google ADK
root_agent = Agent(
    name="weather_time_agent",
    model="gemini-2.0-flash",
    description="시간과 날씨에 대한 질문에 답하는 에이전트",
    instruction="도시의 시간과 날씨에 대한 사용자 질문에 답하는 도움이 되는 에이전트입니다.",
    tools=[get_weather, get_current_time]
)
```

에이전트 프레임워크들은 공통적으로 **Agent + Tools + Instructions** 패턴을 채택하고 있습니다.

In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import asyncio
import os

load_dotenv(override=True)

api_key = os.getenv('OPENAI_API_KEY')
if api_key:
    print("API key found.")
else:
    print("No API key was found — .env 파일에 OPENAI_API_KEY를 설정하세요.")

API key found.
